### 🎯 [미션] 주석이 달린 줄의 None 또는 _______ 부분을 채워 코드를 완성하세요.

각 셀을 순서대로 실행(Shift + Enter)해야 합니다.

---

# 🚨 실시간 안전 경고 시스템

이 노트북에서는 학습된 모델을 사용하여 **공사현장 안전 모니터링 시스템**을 구현합니다.

**시스템 기능:**
1. 이미지에서 안전모 착용/미착용 작업자 탐지
2. 위험 수준 계산 (미착용 비율)
3. 경고 메시지 및 시각화

---
#### 1️⃣ 필요한 라이브러리 설치 및 불러오기

In [ ]:
# 필요한 라이브러리 설치 (최초 1회만 실행)
# - ultralytics: YOLO 모델 사용을 위한 라이브러리
# - koreanize-matplotlib: 그래프에서 한글 폰트 지원
!pip install ultralytics koreanize-matplotlib

In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import koreanize_matplotlib
from PIL import Image as PILImage

from ultralytics import YOLO

print("✅ 라이브러리 로딩 완료!")

---
#### 2️⃣ 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd "/content/drive/MyDrive/2026_AI_Advanced_Study-main/4차시/04_hardhat_detection/code"

---
#### 3️⃣ 학습된 모델 불러오기

In [ ]:
# 학습된 모델의 경로 설정
model_path = './runs/detect/train/weights/best.pt'

# 모델 로드
model = YOLO(model_path)

print(f"✅ 모델 로드 완료: {model_path}")
print(f"   클래스: {model.names}")

---
#### 4️⃣ 안전 상태 분석 함수 구현

이미지에서 탐지된 객체를 분석하여 안전 상태를 반환하는 함수입니다.

**Detection 결과 구조:**
- `result.boxes`: 탐지된 모든 바운딩 박스
- `box.cls`: 클래스 ID
- `box.conf`: 신뢰도
- `box.xyxy`: 좌표 (x1, y1, x2, y2)

In [ ]:
def analyze_safety(result):
    """
    YOLO 예측 결과를 분석하여 안전 상태를 반환합니다.
    
    Args:
        result: YOLO 예측 결과 (단일 이미지)
        
    Returns:
        dict: 안전 분석 결과
    """
    # 🎯 [미션] Detection 결과에서 바운딩 박스를 가져오세요.
    # 힌트: Classification은 probs, Detection은?
    boxes = result._______
    
    # 클래스별 카운트 초기화
    # 0: head (안전모 미착용), 1: helmet (안전모 착용)
    head_count = 0      # 안전모 미착용
    helmet_count = 0    # 안전모 착용
    
    # 각 바운딩 박스 분석
    for box in boxes:
        # 🎯 [미션] 바운딩 박스에서 클래스 ID를 추출하세요.
        # 힌트: box의 어떤 속성에 클래스 정보가 있을까요?
        cls_id = int(box._______[0])
        
        if cls_id == 0:  # head (미착용)
            head_count += 1
        else:  # helmet (착용)
            helmet_count += 1
    
    # 전체 인원 수
    total_count = head_count + helmet_count
    
    # 🎯 [미션] 위험 수준 계산: 미착용 비율
    # 힌트: 미착용자 수 / 전체 인원 수
    if total_count > 0:
        danger_level = _______ / _______
    else:
        danger_level = 0
    
    return {
        'helmet_count': helmet_count,
        'head_count': head_count,
        'total_count': total_count,
        'danger_level': danger_level
    }

print("✅ analyze_safety 함수 정의 완료!")

---
#### 5️⃣ 경고 메시지 생성 함수

In [ ]:
def get_safety_status(danger_level, danger_thresh=0.3):
    """
    위험 수준에 따른 안전 상태를 반환합니다.
    
    Args:
        danger_level: 위험 수준 (0.0 ~ 1.0)
        danger_thresh: 경고 임계값 (자유롭게 설정)
        
    Returns:
        tuple: (상태, 색상, 메시지)
    """
    if danger_level == 0:
        return ('SAFE', (0, 255, 0), '✅ 모든 작업자 안전모 착용!')
    elif danger_level < danger_thresh:
        return ('WARNING', (0, 255, 255), f'⚠️ 일부 미착용자 발견')
    else:
        return ('DANGER', (0, 0, 255), f'🚨 다수 미착용자! 작업 중지 필요!')

print("✅ get_safety_status 함수 정의 완료!")

---
#### 6️⃣ 안전 모니터링 시각화 함수

In [ ]:
def visualize_safety(image_path, model, danger_thresh=0.3):
    """
    이미지에서 안전 상태를 분석하고 시각화합니다.
    """
    # 🎯 [미션] 이미지 예측을 수행하세요.
    # 힌트: model의 어떤 메서드로 예측을 수행할까요?
    results = model._______(image_path, verbose=False)
    result = results[0]
    
    # 안전 분석
    analysis = analyze_safety(result)
    status, color, message = get_safety_status(analysis['danger_level'], danger_thresh)
    
    # 이미지 로드
    img = cv2.imread(image_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    
    # 바운딩 박스 그리기
    for box in result.boxes:
        cls_id = int(box.cls[0])
        conf = float(box.conf[0])
        x1, y1, x2, y2 = map(int, box.xyxy[0])
        
        # 안전모 착용 여부에 따른 색상
        if cls_id == 1:  # helmet (착용)
            box_color = (0, 255, 0)  # 초록
            label = f"Helmet {conf:.0%}"
        else:  # head (미착용)
            box_color = (255, 0, 0)  # 빨강
            label = f"No Helmet {conf:.0%}"
        
        # 박스 그리기
        cv2.rectangle(img, (x1, y1), (x2, y2), box_color, 3)
        cv2.putText(img, label, (x1, y1-10), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, box_color, 2)
    
    # 시각화
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 왼쪽: 탐지 결과
    axes[0].imshow(img)
    axes[0].set_title(f'Detection Result - {status}', fontsize=14)
    axes[0].axis('off')
    
    # 오른쪽: 분석 결과
    axes[1].set_xlim(0, 10)
    axes[1].set_ylim(0, 10)
    axes[1].axis('off')
    
    # 상태 배너
    status_color = {'SAFE': 'green', 'WARNING': 'orange', 'DANGER': 'red'}[status]
    axes[1].text(5, 9, f'🦺 안전 분석 결과', fontsize=18, ha='center', fontweight='bold')
    axes[1].text(5, 7.5, status, fontsize=24, ha='center', fontweight='bold', color=status_color)
    axes[1].text(5, 6, message, fontsize=12, ha='center')
    
    # 통계
    axes[1].text(1, 4, f"🟢 안전모 착용: {analysis['helmet_count']}명", fontsize=14)
    axes[1].text(1, 3, f"🔴 안전모 미착용: {analysis['head_count']}명", fontsize=14)
    axes[1].text(1, 2, f"📊 총 인원: {analysis['total_count']}명", fontsize=14)
    axes[1].text(1, 1, f"⚠️ 위험 수준: {analysis['danger_level']:.1%}", fontsize=14, color=status_color)
    
    plt.tight_layout()
    plt.show()
    
    return analysis

print("✅ visualize_safety 함수 정의 완료!")

---
#### 7️⃣ 시스템 테스트

In [ ]:
# 나만의 위험 임계값 설정 (0.0 ~ 1.0)
# 이 값보다 미착용 비율이 높으면 DANGER로 판단
DANGER_THRESH = 0.3  # 자유롭게 조절해보세요! (예: 0.2, 0.5)

print(f"⚙️ 위험 임계값: {DANGER_THRESH:.0%}")
print(f"   → 미착용 비율이 {DANGER_THRESH:.0%} 이상이면 DANGER 판정\n")

In [ ]:
import glob

# 데모 이미지 테스트
demo_images = glob.glob('../data/demo/*.[jJ][pP][gG]')

print(f"📷 테스트할 이미지: {len(demo_images)}장\n")

for img_path in demo_images:
    print("=" * 60)
    print(f"📁 파일: {os.path.basename(img_path)}")
    print("=" * 60)
    
    analysis = visualize_safety(img_path, model, DANGER_THRESH)
    print()